# Few-shot: Türkiye uyarlaması (A ve B kolları)

İnce defter: bütün mantık `src/hasar` paketinde. Defter git etiketlerini okuyup aşamasını kendi seçer:

- `fewshot-kilidi` etiketi yoksa yalnız başlangıç checkpoint'inin (`ana-708fdc1f5ecb/best.pt`) `sha256` değeri
  hesaplanır ve defter durur (Türkiye'ye dokunan hiçbir adım çalışmaz).
- `fewshot-kilidi` varsa A kolunun ön kontrol + eğitimi, B kolunun ön kontrol + eğitimi ve yalnız ikisi de
  `DONE` olduktan sonra A ve B'nin tek Türkiye test değerlendirmesi çalışır.

Bağlantı koparsa yeniden "Run All" yeterlidir (`last.pt` kaldığı yerden devam eder, `DONE` atlanır,
`test_done.json` varsa test ikinci kez çalışmaz).
Ayrıntı: `docs/protokol.md` §15, `docs/superpowers/specs/2026-11-23-turkiye-few-shot-design.md`.


In [ ]:
# Hücre 1: Drive, aşama seçimi (git etiketleri), kurulum, sırlar
import os
import subprocess
from pathlib import Path

from google.colab import drive, userdata

RUNS = [("configs/fewshot_a.yaml", "fewshot_a"), ("configs/fewshot_b.yaml", "fewshot_b")]

drive.mount("/content/drive")
# Bellek parçalanmasını azaltır; alt süreçler (hasar komutları) de bu ayarı devralır.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
REPO = Path("/content/deprem-hasar-haritasi")
DRIVE = Path("/content/drive/MyDrive/deprem-hasar")
DATA = Path("/content/data/bright")
CKPT = Path("/content/data/checkpoints")
OUT = DRIVE / "kosular"
RES = DRIVE / "sonuclar" / "fewshot"


def sh(cmd, check=True):
    """Run a shell command and stream its output into the cell (Colab hides plain subprocess output)."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if check and rc:
        raise subprocess.CalledProcessError(rc, cmd)
    return rc


if not REPO.exists():
    sh(f"git clone https://github.com/OgzhnTarhn/deprem-hasar-haritasi.git {REPO}")
sh(f"cd {REPO} && git fetch --tags --force --quiet origin")
# Önceki oturumun klonu kirli kalmış olabilir; etiketli commit'e geçmeden önce sıfırla.
sh(f"cd {REPO} && git reset --quiet --hard")
tags = subprocess.run(f"cd {REPO} && git tag -l", shell=True, capture_output=True, text=True).stdout
TAGS = {line.strip() for line in tags.splitlines() if line.strip()}
LOCKED = "fewshot-kilidi" in TAGS
REF = "origin/main"
sh(f"cd {REPO} && git checkout --quiet --force {REF}")
dirty = subprocess.run(f"cd {REPO} && git status --porcelain", shell=True, capture_output=True, text=True).stdout
assert not dirty.strip(), "Çalışma ağacı kirli; commit değiştirilmemeli."


def locked_files_changed(tag, paths):
    """Return the files under ``paths`` that differ between ``tag`` and HEAD (or a note if not an ancestor)."""
    anc = subprocess.run(f"cd {REPO} && git merge-base --is-ancestor {tag} HEAD", shell=True).returncode
    if anc != 0:
        return [f"{tag} HEAD'in atası değil"]
    out = subprocess.run(
        f"cd {REPO} && git diff --name-only {tag} HEAD -- {paths}", shell=True, capture_output=True, text=True
    ).stdout
    return [line for line in out.splitlines() if line.strip()]


if LOCKED:
    changed = locked_files_changed(
        "fewshot-kilidi",
        "docs/protokol.md configs/fewshot_a.yaml configs/fewshot_b.yaml results/fewshot/bolme.json",
    )
    assert not changed, f"fewshot-kilidi'nden sonra kilitli dosya değişmiş: {changed}"
COMMIT = subprocess.run(f"cd {REPO} && git rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip()
STAGE = "ön kontrol + eğitim (A, B) + Türkiye testi" if LOCKED else "yalnız başlangıç sha256 (kilit bekleniyor)"
print("=" * 72)
print("AŞAMA:", STAGE)
print("REF:", REF)
print("commit:", COMMIT)
print("=" * 72)
# Colab'ın torch ve timm sürümleri kullanılır; torch yeniden kurulmaz.
sh(f"pip install -q -e {REPO} einops")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN sırrı yok; indirme anonim yapılır.")
DRIVE.mkdir(parents=True, exist_ok=True)

In [ ]:
# Hücre 2: Ortam kapısı (A100 ya da L4)
import shutil

import torch

props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 2**30, 1), "GB")
# Colab A100 seçilse de bazen L4 (22 GB) veriyor; ikisi de bf16 destekler.
assert any(gpu in props.name for gpu in ("A100", "L4")), f"A100 ya da L4 gerekli, gelen: {props.name}"
free_gb = shutil.disk_usage("/content").free / 2**30
assert free_gb >= 60, f"Disk yetersiz: {free_gb:.0f} GB"
print("torch", torch.__version__, "CUDA", torch.version.cuda)

In [ ]:
# Hücre 3: Ana koşunun checkpoint'i (sha256), kilit kapısı ve veri
ANA_SRC = OUT / "ana-708fdc1f5ecb" / "best.pt"
PRETRAINED = CKPT / "ana_best.pt"
if not PRETRAINED.exists():
    assert ANA_SRC.is_file(), f"Ana koşunun checkpoint'i bulunamadı: {ANA_SRC}"
    CKPT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ANA_SRC, PRETRAINED)
sh(f"cd {REPO} && hasar sha256 {PRETRAINED}")


class KilitBekleniyor(RuntimeError):
    """fewshot-kilidi etiketi atılana kadar eğitimi ve testi durdurur."""


if not LOCKED:
    print("Bu sha256 değerini Claude'a gönderin; yapılandırmalara yazılıp fewshot-kilidi atılacak.")
    raise KilitBekleniyor("fewshot-kilidi etiketi yok: eğitim ve Türkiye testi çalışmaz.")

sh(f"cd {REPO} && hasar indir --root {DATA} --cache /content/data/cache")  # .done varsa atlanır

In [ ]:
# Hücre 4: Ön kontrol ve eğitim (her kol için)
import json

REPRO = REPO / "results" / "yeniden-uretim"
for key in ["siamcrnn", "unet", "damageformer"]:
    assert (REPRO / f"{key}_test.json").is_file(), f"Yeniden üretim kaydı eksik: {REPRO / f'{key}_test.json'}"


def run_arm(config, name):
    pre_dir = OUT / "onkontrol" / name
    pre = pre_dir / "preflight.json"
    # Ön kontrol zaten geçtiyse kendini atlar; bu yüzden her "Run All"da çağrılır.
    rc = sh(
        f"cd {REPO} && hasar on-kontrol --config {config} --data {DATA} --out {pre_dir} "
        f"--device cuda --pretrained {PRETRAINED} --repro-dir {REPRO}",
        check=False,
    )
    failed = []
    if pre.exists():
        report = json.loads(pre.read_text(encoding="utf-8"))
        for gate in report.get("gates", []):
            verdict = "geçti" if gate.get("passed") else "KALDI"
            print(f"kapı {gate.get('gate')}  {verdict}  {gate.get('detail', '')}")
            if not gate.get("passed"):
                failed.append(gate.get("gate"))
            if gate.get("gate") == 9:
                print("kırpma eşiği (clip):", gate.get("value", {}).get("clip"))
    if rc != 0:
        raise AssertionError(
            f"Ön kontrol kapılarından biri geçmedi ({name}, kapılar: {failed}); eğitim başlamaz.\n"
            f"preflight.json: {pre}\nBu dosyayı Claude'a gönderin."
        )
    # last.pt varsa devam eder, DONE varsa atlanır.
    sh(
        f"cd {REPO} && hasar egit --config {config} --data {DATA} --out {OUT} "
        f"--device cuda --pretrained {PRETRAINED} --commit {COMMIT} --preflight {pre}"
    )

In [ ]:
# Hücre 5: A kolu (sınıf ağırlığı yok)
run_arm(*RUNS[0])

In [ ]:
# Hücre 6: B kolu (karekök sınıf ağırlıkları)
run_arm(*RUNS[1])

In [ ]:
# Hücre 7: Türkiye test değerlendirmesi (yalnız iki kol da DONE olduktan sonra, kol başına bir kez)
INST = Path("/content/data/ornekler")
sh(f"cd {REPO} && hasar ornekler --dest {INST} --cache /content/data/cache")
ARMS = " ".join(config for config, _ in RUNS)
for config, name in RUNS:
    out = RES / name.removeprefix("fewshot_")  # a, b: hasar fewshot-kiyas varsayılan klasörleri
    if (out / "test_done.json").exists():
        print(f"{name}: test zaten yapıldı, ikinci kez çalışmaz.")
    else:
        # Komut, kollardan biri DONE değilse kendisi reddeder (tasarım §9.4).
        sh(
            f"cd {REPO} && hasar fewshot-test --config {config} --arms {ARMS} "
            f"--data {DATA} --runs {OUT} --out {out} --instances {INST} --device cuda"
        )
print("Sonuçlar:", RES)

In [ ]:
# Hücre 8: Kayıtları göster ve çalışma zamanını bırak (CU yakımı durur)
from google.colab import runtime

done = sorted(OUT.glob("fewshot_*/DONE"))
print("DONE:", [str(p) for p in done])
for p in done:
    print(json.loads(p.read_text(encoding="utf-8")))
tests = sorted(RES.glob("*/test_done.json"))
print("TEST DONE:", [str(p) for p in tests])
for p in tests:
    print(json.loads(p.read_text(encoding="utf-8")))
runtime.unassign()

Sonra: Drive'dan `sonuclar/fewshot` klasörünü indirip Claude'a verin (`hasar fewshot-kiyas` yerelde çalışır).
